# Academy Award Statistical Prediction Model

A prediction model built on historical Academy Award data. It looks at common precursor awards (PGA, DGA, SAG, BAFTA, Golden Globes, Critics' Choice) leading up to the Oscars and calculates how often each one's winner has gone on to win the Oscar itself. It then uses those win-rates to score and predict this year's nominees.

In [1]:
import pandas

In [2]:
# Create Data Frames for all Categories
best_picture_df = pandas.read_csv("awards_historical_data/best_picture_history.csv")
best_actor_df = pandas.read_csv("awards_historical_data/best_actor_history.csv")
best_actress_df = pandas.read_csv("awards_historical_data/best_actress_history.csv")
best_supporting_actor_df = pandas.read_csv("awards_historical_data/best_supporting_actor_history.csv")
best_supporting_actress_df = pandas.read_csv("awards_historical_data/best_supporting_actress_history.csv")

In [3]:
# ── Statistics: historical precursor win-rates for Best Picture ──
# For each precursor award, this measures the rate at which its winners
# historically went on to win the Oscar too. Those rates become the
# weights used in best_picture_score() below.
# Assemble all stats for Best Picture
def get_best_picture_statistics(best_picture_df):

  # Gather PGA Statistics
  pga_winners = best_picture_df[best_picture_df['won_pga'] == 1]
  pga_to_oscar_rate = pga_winners['won_oscar'].mean()

  # Gather DGA Statistics
  dga_winners = best_picture_df[best_picture_df['won_dga'] == 1]
  dga_to_oscar_rate = dga_winners['won_oscar'].mean()

  # Gather BAFTA Statistics
  bafta_winners = best_picture_df[best_picture_df['won_bafta'] == 1]
  bafta_to_oscar_rate = bafta_winners['won_oscar'].mean()

  # Gather Critic's Choice Statistics
  critics_winners = best_picture_df[best_picture_df['won_critics'] == 1]
  critics_to_oscar_rate = critics_winners['won_oscar'].mean()

  # Gather Average Nominations for Winners
  # (not a rate like the ones above - just reference numbers used to
  # sanity-check nomination_count_weight further down)
  oscar_winners = best_picture_df[best_picture_df['won_oscar'] == 1]
  oscar_winner_average_nomination_count = oscar_winners['total_oscar_nominations'].mean()

  # Gather Average Nominations for Non-Winners
  non_oscar_winners = best_picture_df[best_picture_df['won_oscar'] == 0]
  non_oscar_winner_average_nomination_count = non_oscar_winners['total_oscar_nominations'].mean()

  return {"pga_rate" : pga_to_oscar_rate,
          "dga_rate" : dga_to_oscar_rate,
          "bafta_rate" : bafta_to_oscar_rate,
          "critics_rate" : critics_to_oscar_rate,
          "average_winner_nominations" : oscar_winner_average_nomination_count,
          "average_non_winner_nominations" : non_oscar_winner_average_nomination_count}


In [4]:
# ── Statistics: same idea, for the acting categories ──
# Mirrors get_best_picture_statistics above, just swapping in the acting
# categories' own precursor awards (SAG, Golden Globes) in place of PGA/DGA.
# Assemble all stats for acting categories
def get_acting_category_statistics(category_df):

  # Gather SAG Statistics
  sag_winners = category_df[category_df['won_sag'] == 1]
  sag_to_oscar_rate = sag_winners['won_oscar'].mean()

  # Gather Globe Statistics
  globe_winners = category_df[category_df['won_globe'] == 1]
  globe_to_oscar_rate = globe_winners['won_oscar'].mean()

  # Gather BAFTA Statistics
  bafta_winners = category_df[category_df['won_bafta'] == 1]
  bafta_to_oscar_rate = bafta_winners['won_oscar'].mean()

  # Gather Critic's Choice Statistics
  critics_winners = category_df[category_df['won_critics'] == 1]
  critics_to_oscar_rate = critics_winners['won_oscar'].mean()

  # Gather Average Nominations for Winners
  oscar_winners = category_df[category_df['won_oscar'] == 1]
  oscar_winner_average_nomination_count = oscar_winners['total_oscar_nominations'].mean()

  # Gather Average Nominations for Non-Winners
  non_oscar_winners = category_df[category_df['won_oscar'] == 0]
  non_oscar_winner_average_nomination_count = non_oscar_winners['total_oscar_nominations'].mean()

  return {"sag_rate" : sag_to_oscar_rate,
          "globe_rate" : globe_to_oscar_rate,
          "bafta_rate" : bafta_to_oscar_rate,
          "critics_rate" : critics_to_oscar_rate,
          "average_winner_nominations" : oscar_winner_average_nomination_count,
          "average_non_winner_nominations" : non_oscar_winner_average_nomination_count}

With Best Picture, historical data determined that the highest predictor award is the Producers' Guild Awards (PGA) at ~72% of winners receiving the Oscar. From there, it is followed by a tie. The Director's Guild Awards (DGA) resulted in ~54% of winners receiving the Oscar, this is the same for the Critic's Choice Awards. Lastly, the BAFTA Awards resulted in ~18% of winners receiving the Oscar.

The amount of film nominations was included to design a feature that helped represent the film's support in the community. The results for Best Picture showed that Oscar winners had around 7.18 nominations while non Oscar winners had around 6.54 nominations. This category will contribute very lightly to the overall model as it is not a main contributor, with both sides having very similar results.

In [5]:
# Run the two functions above once per category, so every category gets
# its own independent set of precursor rates
# Gather all statistics for categories
picture_stats = get_best_picture_statistics(best_picture_df)
actor_stats = get_acting_category_statistics(best_actor_df)
actress_stats = get_acting_category_statistics(best_actress_df)
supporting_actor_stats = get_acting_category_statistics(best_supporting_actor_df)
supporting_actress_stats = get_acting_category_statistics(best_supporting_actress_df)

Now, for the nomination weight. It is important to consider that the number of nominations had a very small impact on the overall outcome of the award. There were cases in which winners had the highest nomination count and other in which winners had the smallest nomination count. The final statistics also showed very little variation between winner and non winner nomination counts, with the highest being around ~2 fewer nominations for non-winners.

Thus, a choice was made to hardcode the weight for nomination counts. This is a design choice being made at the moment but could ultimately change if the model is tuned or the feature is removed in its entirety. For now, it shall remain as an indicator of support in the academy for the film.

In [6]:
# Hand-picked rather than derived as a rate like the precursor awards above -
# see the reasoning in the markdown cell just above this one.
nomination_count_weight = 0.04

In [7]:
# ── Scoring: weight a single nominee using the rates computed above ──
# Calculates the score of a Best Picture Nominee
def best_picture_score(nominee_row, statistics):

  score = 0

  # nominee_row['won_pga'] is 0 or 1, so a precursor only adds its rate to
  # the score if this specific nominee actually won it
  score += nominee_row['won_pga'] * statistics['pga_rate']
  score += nominee_row['won_dga'] * statistics['dga_rate']
  score += nominee_row['won_bafta'] * statistics['bafta_rate']
  score += nominee_row['won_critics'] * statistics['critics_rate']
  score += nominee_row['total_oscar_nominations'] * nomination_count_weight

  return score

In [8]:
# Same idea as best_picture_score above, just with the acting categories'
# own guild columns in place of PGA/DGA
# Calculates the score of a nominee in an Acting Category
def acting_score(nominee_row, statistics):
  score = 0

  score += nominee_row['won_sag'] * statistics['sag_rate']
  score += nominee_row['won_globe'] * statistics['globe_rate']
  score += nominee_row['won_bafta'] * statistics['bafta_rate']
  score += nominee_row['won_critics'] * statistics['critics_rate']
  score += nominee_row['total_oscar_nominations'] * nomination_count_weight

  return score

In [9]:
# ── Prediction: score every nominee in the field, return the top one ──
# Obtains Prediction for Best Picture
def predict_picture(path,stats):
  picture_df = pandas.read_csv(path)
  nominee_scores = []

  # Score every nominee one row at a time, then sort and keep the highest
  for i,row in picture_df.iterrows():
    score = best_picture_score(row,stats)
    score = {'film':row['film'], 'score':score}
    nominee_scores.append(score)

  nominee_scores_df = pandas.DataFrame(nominee_scores).sort_values(by='score',ascending=False)
  best_picture_prediction = nominee_scores_df['film'].head(1).to_string(index=False)

  return best_picture_prediction

In [10]:
# Same idea as predict_picture above, for the acting categories
def predict_acting(path,stats):
  picture_df = pandas.read_csv(path)
  nominee_scores = []

  for i,row in picture_df.iterrows():
    score = acting_score(row,stats)
    score = {'actor':row['actor_name'], 'score':score}
    nominee_scores.append(score)

  nominee_scores_df = pandas.DataFrame(nominee_scores).sort_values(by='score',ascending=False)
  acting_prediction = nominee_scores_df['actor'].head(1).to_string(index=False)

  return acting_prediction

## Running the 2026 Predictions

With the statistics and scoring functions set up for every category, we can now run the model against the 2026 nominees and see who it predicts will win.

In [13]:
# ── Run predictions for the 2026 field and show the results ──
best_actor = predict_acting("2026_oscar_nominees/best_actor_2026.csv",actor_stats)
best_actress = predict_acting("2026_oscar_nominees/best_actress_2026.csv", actress_stats)
best_support_actor = predict_acting("2026_oscar_nominees/best_supporting_actor_2026.csv", supporting_actor_stats)
best_support_actress = predict_acting("2026_oscar_nominees/best_supporting_actress_2026.csv", supporting_actress_stats)
best_picture = predict_picture("2026_oscar_nominees/best_picture_2026.csv", picture_stats)


print("🏆 2026 ACADEMY AWARDS PREDICTED WINNERS 🏆\n")

print(f"🎭 Best Actor in a Supporting Role   :  {best_support_actor}")
print(f"🎭 Best Actress in a Supporting Role :  {best_support_actress}")
print(f"🎭 Best Actor in a Leading Role      :  {best_actor}")
print(f"🎭 Best Actress in a Leading Role    :  {best_actress}")
print(f"🎬 Best Picture                      :  {best_picture}")


🏆 2026 ACADEMY AWARDS PREDICTED WINNERS 🏆

🎭 Best Actor in a Supporting Role   :  Sean Penn
🎭 Best Actress in a Supporting Role :  Amy Madigan
🎭 Best Actor in a Leading Role      :  Michael B. Jordan
🎭 Best Actress in a Leading Role    :  Jessie Buckley
🎬 Best Picture                      :  One Battle After Another
